# Stage 1 - Preprocessing & Validation Setup

This notebook contains:
1. **Environment & Paths**: Verify working directories and imports
2. **Dynamic Random Sampling**: Inspect random rows from the multi-million record TSV files instantly
3. **Text & Address Normalization**: Test name suffix stripping, address expansion, and French accents
4. **Local Validation Split**: Create a fast, balanced 10,000-query validation slice with ground truth
5. **Validation Verification**: Verify singleton ratio and entity matching structures
6. **Full Dataset Preprocessing**: Batch-process all train and test files into `dataset/preprocessed/`

In [7]:
# 1. Environment & Path Setup
import os
import sys
import random
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(f"Python Version : {sys.version.split()[0]}")
print(f"Pandas Version : {pd.__version__}")
print(f"Root Directory : {ROOT_DIR}")

Python Version : 3.14.3
Pandas Version : 3.0.6
Root Directory : c:\Users\satya\Downloads\amazon


In [8]:
# 2. Fast Dynamic Random Sampling from Dataset
def get_random_sample(filepath, n=5):
    """Pick n random rows from a large TSV file instantly using random byte seeks."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        header = f.readline().strip().split("\t")
        file_size = os.path.getsize(filepath)
        rows = []
        for _ in range(n * 5):
            pos = random.randint(0, max(0, file_size - 2000))
            f.seek(pos)
            f.readline()  # discard partial line
            line = f.readline().strip()
            if line:
                parts = line.split("\t")
                if len(parts) == len(header):
                    rows.append(parts)
            if len(rows) >= n:
                break
    return pd.DataFrame(rows[:n], columns=header)

DATA_DIR = "../student_resource/dataset/train"

print("Random Source 1 sample (changes every run):")
s1_sample = get_random_sample(f"{DATA_DIR}/train_source1.tsv", n=5)
display(s1_sample)

print("\nRandom Ground Truth sample (changes every run):")
gt_sample = get_random_sample(f"{DATA_DIR}/train_ground_truth.tsv", n=5)
display(gt_sample)

Random Source 1 sample (changes every run):


,entity_id,business_name,business_address,country
0,S1-386065111,Saru Innovations Private Limited,"Kamika 14/2, New Palasia, Indore, Madhya Pradesh",India
1,S1-582704379,Choice Workshop,"2863 Powell Drive Extension, Valdese, NC",US
2,S1-535635757,Csd Brothers Private Limited,"Durlabhmani6-Diwanpara, Rajkot, Gujarat",India
3,S1-179185018,Myers Impact,"92 Neptune Street, Lynn, MA",US
4,S1-738295007,Santacruz (W) Infrabuild Private Limited,"24, Linking Road, Maharashtra, Mumbai, Usha Vi...",India



Random Ground Truth sample (changes every run):


,source1_entity_id,matched_entity_ids
0,S1-779774356,"S2-472787047,S2-314703071,S2-736054125,S2-4376..."
1,S1-692916146,"S2-321155310,S2-634087491,S3-117729847"
2,S1-855460029,"S2-464247011,S2-343792599,S3-751625383,S3-4246..."
3,S1-15327943,"S3-809660634,S3-328952504,S3-129614972"
4,S1-293914793,"S2-217973832,S2-618263332,S3-557986926,S3-7459..."


In [9]:
# 3. Test Preprocessing Pipeline on Random Real Records + France Test Case
from src.preprocessing import preprocess_record

# Real records sampled dynamically from Cell 2
real_records = s1_sample.to_dict("records")

# Include a French test case (from test set distribution)
france_sample = {
    "entity_id": "TEST-FR-01",
    "business_name": "Société Générale S.A.R.L.",
    "business_address": "12 Bd Haussmann, 75009 Paris",
    "country": "France",
}
all_samples = real_records + [france_sample]

print(f"=== Preprocessing {len(all_samples)} Records (Dynamic Real Samples + French Test Case) ===\n")
for s in all_samples:
    cleaned = preprocess_record(s)
    print(f"ID: {s['entity_id']} ({s.get('country', '')})")
    print(f"  Original : {s.get('business_name', '')} | {s.get('business_address', '')}")
    print(f"  Cleaned  : {cleaned['clean_name']} [Core: {cleaned['core_name']}] (Suffix: '{cleaned['legal_suffix']}')")
    print(f"  Address  : {cleaned['clean_address']}")
    print(f"  Postal   : {cleaned['postal_code']}")
    print("-" * 80)

=== Preprocessing 6 Records (Dynamic Real Samples + French Test Case) ===

ID: S1-386065111 (India)
  Original : Saru Innovations Private Limited | Kamika 14/2, New Palasia, Indore, Madhya Pradesh
  Cleaned  : saru innovations private limited [Core: saru innovations] (Suffix: 'private limited')
  Address  : kamika 14 2 new palasia indore madhya pradesh
  Postal   : 
--------------------------------------------------------------------------------
ID: S1-582704379 (US)
  Original : Choice Workshop | 2863 Powell Drive Extension, Valdese, NC
  Cleaned  : choice workshop [Core: choice workshop] (Suffix: '')
  Address  : 2863 powell drive extension valdese nc
  Postal   : 
--------------------------------------------------------------------------------
ID: S1-535635757 (India)
  Original : Csd Brothers Private Limited | Durlabhmani6-Diwanpara, Rajkot, Gujarat
  Cleaned  : csd brothers private limited [Core: csd brothers] (Suffix: 'private limited')
  Address  : durlabhmani6 diwanpara rajkot 

In [10]:
# 4. Generate Fast Stratified Local Validation Split (10,000 S1 queries)
!python ../scripts/create_val_split.py --data-dir ../student_resource/dataset/train --out-dir ../dataset/val_split --num-s1 10000 --num-distractors 30000

Loading ground truth from ../student_resource/dataset/train/train_ground_truth.tsv ...
Selected 10,000 Source 1 entities.
Ground truth requires 16,687 S2 matches and 17,945 S3 matches.
Saved val_ground_truth.tsv
Filtering Source 1...
Saved val_source1.tsv with 10,000 rows.
Filtering Source 2...
Saved val_source2.tsv with 526,126 rows.
Filtering Source 3...
Saved val_source3.tsv with 557,315 rows.

Validation split creation complete in: ../dataset/val_split


In [11]:
# 5. Verify the Generated Validation Dataset
val_s1 = pd.read_csv("../dataset/val_split/val_source1.tsv", sep="\t")
val_gt = pd.read_csv("../dataset/val_split/val_ground_truth.tsv", sep="\t")

singletons = (val_gt["matched_entity_ids"].isna() | (val_gt["matched_entity_ids"] == "")).sum()
print(f"Validation Source 1 rows    : {len(val_s1):,}")
print(f"Validation Ground Truth rows : {len(val_gt):,}")
print(f"Singletons (no match)        : {singletons:,} ({singletons / len(val_gt) * 100:.1f}%)")
display(val_s1.head(3))
display(val_gt.head(3))

Validation Source 1 rows    : 10,000
Validation Ground Truth rows : 10,000
Singletons (no match)        : 560 (5.6%)


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-68456921,Dynamic Hospitality Private Limited,"2Nd Floor, Trivedi Bhawan, Pghpl Group Campus,...",India
2,S1-874413333,Primary Care Classic Associates LLC,"James City County, VA, 3005 London Company Way",US


,source1_entity_id,matched_entity_ids
0,S1-508337570,"S2-582025035,S3-529082874"
1,S1-878749336,"S2-790690416,S2-733913751,S3-932487174"
2,S1-49527993,"S2-860744957,S2-39902340,S2-237585210,S3-82327..."


### 6. Preprocess Complete Dataset (Train & Test)

Runs streaming chunked normalization across all 6 files:
- `train_source1.tsv`, `train_source2.tsv`, `train_source3.tsv`
- `test_source1.tsv`, `test_source2.tsv`, `test_source3.tsv`

Outputs are saved in `dataset/preprocessed/` with clean names, core names, legal suffixes, standardized addresses, and postal codes.

In [12]:
# Run high-throughput streaming preprocessing across the complete dataset
!python ../scripts/preprocess_dataset.py --input-dir ../student_resource/dataset --output-dir ../dataset/preprocessed --split all --chunksize 150000

=== Starting Preprocessing (Split: all) ===
Input base : ../student_resource/dataset
Output base: ../dataset/preprocessed
Total files: 6
Format     : tsv | workers: 1

Processing train_source1.tsv -> train_source1.tsv ...
Completed 2,206,821 records in 337.6s (6,536 records/s).

Processing train_source2.tsv -> train_source2.tsv ...
Completed 5,034,616 records in 829.7s (6,068 records/s).

Processing train_source3.tsv -> train_source3.tsv ...
Completed 5,285,603 records in 811.2s (6,516 records/s).

Processing test_source1.tsv -> test_source1.tsv ...
Completed 1,732,544 records in 272.7s (6,352 records/s).

Processing test_source2.tsv -> test_source2.tsv ...
Completed 4,887,273 records in 759.1s (6,439 records/s).

Processing test_source3.tsv -> test_source3.tsv ...
Completed 5,082,316 records in 818.3s (6,211 records/s).

All tasks finished in 63.8 minutes!



train_source1.tsv: 0chunk [00:00, ?chunk/s]
train_source1.tsv: 1chunk [00:21, 21.78s/chunk]
train_source1.tsv: 2chunk [00:42, 21.32s/chunk]
train_source1.tsv: 3chunk [01:03, 21.05s/chunk]
train_source1.tsv: 4chunk [01:24, 20.98s/chunk]
train_source1.tsv: 5chunk [01:45, 20.90s/chunk]
train_source1.tsv: 6chunk [02:07, 21.23s/chunk]
train_source1.tsv: 7chunk [02:28, 21.25s/chunk]
train_source1.tsv: 8chunk [02:49, 21.21s/chunk]
train_source1.tsv: 9chunk [03:10, 21.10s/chunk]
train_source1.tsv: 10chunk [03:32, 21.40s/chunk]
train_source1.tsv: 11chunk [03:54, 21.48s/chunk]
train_source1.tsv: 12chunk [04:20, 23.09s/chunk]
train_source1.tsv: 13chunk [04:48, 24.34s/chunk]
train_source1.tsv: 14chunk [05:14, 24.94s/chunk]
train_source1.tsv: 15chunk [05:37, 24.42s/chunk]
train_source1.tsv: 15chunk [05:37, 22.51s/chunk]

train_source2.tsv: 0chunk [00:00, ?chunk/s]
train_source2.tsv: 1chunk [00:30, 30.84s/chunk]
train_source2.tsv: 2chunk [00:54, 26.49s/chunk]
train_source2.tsv: 3chunk [01:18, 25.23